In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## $\text{\textcolor{green}{Preprocessing data}}$

In [ ]:
df_temp = pd.read_csv(ROOT / "data/complete_dataframe2_30min.csv",delimiter=",",decimal=".",parse_dates=["ts"],index_col="ts")
df_load = pd.read_csv(ROOT / "data/archive/archive/building_consumption.csv", parse_dates=["timestamp"])
df_price = pd.read_csv(ROOT / "data/DayAheadData_fulltimeperiod.csv")
df_price["ts"] = pd.to_datetime(df_price['ts'])
df_price = df_price.set_index("ts") # set time as index



filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14") &
    (df_load["timestamp"] <= "2022-04-11")
]
# How many 15-min readings are there available per meter_id?
counts = filtered.groupby("meter_id").size().reset_index(name="num_readings")

In [ ]:
df_temp2 = df_temp.join(df_price,how="inner")
df_temp2 = df_temp2[:-1]

In [ ]:
downsizing_factor = 0.02


total_load = filtered.groupby("timestamp")["consumption"].sum().reset_index(name="total_consumption")
total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])+pd.DateOffset(years=4)
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean() * downsizing_factor

In [ ]:
df = df_temp2.join(total_load_30min,how="inner")

In [ ]:
# Confirm data has 48-step days
total_load_30min["date"] = total_load_30min.index.date
load_counts = total_load_30min.groupby("date").size()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[total_load_30min["date"].isin(load_complete_days)].copy()

# Make days into 48-step vectors

load_day_profiles = []

for _, day_df in load_days_df.groupby("date"):
    day_profile = day_df.sort_index()["total_consumption"].to_numpy()
    load_day_profiles.append(day_profile)

# Visualization of daily profiles <-- Would the MW scale be per time step or across the day profile?

daily_totals = total_load_30min.groupby(total_load_30min.index.date)["total_consumption"].sum()
daily_energy = daily_totals * 0.5

In [ ]:
# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg["Wind_total"] = df_agg[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)
df_agg = df_agg.drop(columns=["Aircon_WT Power_m", "Gaia_WT Power_m","B117_m", "B319_m", "B330_1_m", "B330_2_m",
                               "B330_3_m", "B716_m", "B715_m","wind_dir","wind_max","wind_min","wind_speed","radia_glob","temp_dry"])
# Keep only complete days
df_agg["date"] = df_agg.index.date
counts_per_day = df_agg.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index
df_agg = df_agg[df_agg["date"].isin(complete_days)].copy()

## $\text{\textcolor{green}{Generation vs Load}}$

In [ ]:
df_grouped = {str(date): group for date, group in df_agg.groupby("date")}

In [ ]:
## PLOT DAILY PROFILE OF LOAD AND POWER 
days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]
cols = ["PV_total","Wind_total","total_consumption"]
# Calculate number of subplots needed
n_days = len(days)
n_cols = 1
n_rows = 4

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 10))
axes = axes.flatten()  # Flatten to easily index

# Plot each day
for i, day in enumerate(days):
    df_day = df_grouped[day]  
    
    # Plot different parameters (columns) vs time
    # Assuming 'timestamp' or 'time' is your x-axis column
    for column in cols:
        axes[i].plot(df_day.index.strftime('%H:%M'), df_day[column], label=column, marker='o')
    
    axes[i].set_title(f'Day: {day}')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('kW')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='x', rotation=45)

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## $\text{\textcolor{green}{Implementing the solver with "Perfect Knowledge" }}$

In [ ]:
# Daily solver
def solve_day_profile(df, minimum_self_sufficiency, E_capacity, P_capacity, charge_eff, discharge_eff, dT, C_rate):
    PV_gen = df["PV_total"].to_numpy()
    Wind_gen = df["Wind_total"].to_numpy()
    n = len(df)
    load = df["total_consumption"].to_numpy()
    price = df["DayAheadPriceDKK"].to_numpy()
    
    if P_capacity > C_rate * E_capacity:
        print(f"Warning: P_capacity={P_capacity}kW exceeds C_rate * E_capacity = {C_rate * E_capacity}")

    # Constants
    Pload = cp.Constant(load)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)
    P_capacity = cp.Constant(P_capacity)
    E_capacity = cp.Constant(E_capacity)

    # Variables

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)

    Pgrid_buy = cp.Variable(n) # if positive, energy is being bought from the grid. if negative we are selling energy to the grid. 

    SOC = cp.Variable(n)

    constraints = [
        Pcharge <= P_capacity, #### should it include the mask z or not? 
        Pdischarge <= P_capacity,

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity # Do we want this?  
        ]


    cost_energy_buy = cp.sum(cp.multiply(Pgrid_buy, price)) * dT 
    #DKK 
    

    problem = cp.Problem(cp.Minimize(cost_energy_buy), constraints)
    
    try:
        problem.solve(solver=cp.OSQP, verbose=True,max_iter=1000000)



        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"Problem arrose: {problem.status}")
            return {
            "status": problem.status,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,

            "Pgrid_buy": np.nan,
            "Pgrid_sell": np.nan,
            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
            }
        
        Pgrid_sell = cp.neg(Pgrid_buy)
        Pgrid_buy = cp.pos(Pgrid_buy)

        return {
        
            "status": problem.status,
            "Total_cost_DKK": (cost_energy_buy).value,
            "Cost_of_bought_energy_DKK": cp.sum(cp.multiply(Pgrid_buy, price)) * dT,
            "bought_energy_kWh": np.sum(Pgrid_buy.value) * dT,
            "sold_energy_kWh": np.sum(Pgrid_sell.value) * dT,

            "Pgrid_buy": Pgrid_buy.value,
            "Pgrid_sell": Pgrid_sell.value,
            "Pcharge": Pcharge.value,
            "Pdischarge": Pdischarge.value,
            "SOC": SOC.value
        }

    except Exception as e:
        print(f"Error arose {type(e).__name__}")
        return {
            "status": type(e).__name__,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,

            "Pgrid_buy": np.nan,
            "Pgrid_sell": np.nan,
            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
        }

## $\text{\textcolor{green}{Running Solver}}$

In [ ]:
# Settings
dT = 0.5
charge_eff = 0.95
discharge_eff = 0.95
C_rate = 1
P_capacity = 30
E_capacity = 120


min_self_sufficiency = 0 # NO REQUIREMENT

In [ ]:
# Run the Optimization for the given days

days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]

def operational_schedule(df,days:list, E_capacity, P_capacity,charge_eff,discharge_eff):
    profiles = []
    
    for day in days:
        print("---------------------------------------------------------")
        print("Optimization starting for ", day)
        print("---------------------------------------------------------")

        day_df = df[day]

        out = solve_day_profile(
            day_df,
            min_self_sufficiency,
            E_capacity,
            P_capacity,
            charge_eff,
            discharge_eff,
            dT,
            C_rate
        )

        # Build time-series dataframe
        profile_df = pd.DataFrame({
            "ts": day_df.index,
            "P_buy": out["Pgrid_buy"],
            "P_sell": out["Pgrid_sell"],
            "P_charge": out["Pcharge"],
            "P_discharge": out["Pdischarge"],

            "SOC": out["SOC"],
            "price": day_df["DayAheadPriceDKK"].values,
            "load": day_df["total_consumption"].values,
            "PV": day_df["PV_total"].values,
            "Wind": day_df["Wind_total"].values,
        })

        profile_df["date"] = day
        print(f"Status for the optimization for {day}: {out["status"]} ")
        print(f"Total cost is: {out["Total_cost_DKK"]}")
        print(f"Amount of bought energy is: {out["bought_energy_kWh"]}")
        print(f"Amount of sold energy is: {out["sold_energy_kWh"]}")

        profiles.append(profile_df)


    profiles_df = pd.concat(profiles, ignore_index=True)
    profiles_df = profiles_df.set_index("ts")
    return profiles_df

## $\text{\textcolor{green}{Analyzing Solver Output}}$

In [ ]:
# Plot the Daily Power Profiles
def plot_like11(profile_df,days:list):
    pos_cols = ["PV",	"Wind", "P_discharge", "P_buy"]
    pos_color = ["gold","lightskyblue","plum","lightgreen"]
    labels = ["PV", "Wind", "Battery", "Grid"]

    neg_cols = ["P_charge","P_sell"]
    neg_color = ["plum","lightgreen"]

    n_days = len(days)
    n_cols = 2
    n_rows = 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 9))

    axes = axes.flatten()  # Flatten to easily index


    for j, day in enumerate(days):

        day_plot = profile_df[profile_df["date"] == day]


        width = 0.85/len(day_plot) # the width of the bars: can also be len(x) sequence


        
        top = np.zeros(len(day_plot))
        bottom = np.zeros(len(day_plot))
        for i,col in enumerate(pos_cols): # Plotting all "positive" power
            color = pos_color[i]
            axes[j].bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color = color)

            top += day_plot[col]

        for i,col in enumerate(neg_cols): # Plotting all "negative" power
            color = neg_color[i]
            axes[j].bar(day_plot.index, - day_plot[col], width, bottom=bottom, color=color)

            bottom -= day_plot[col]

        axes[j].scatter(day_plot.index,day_plot["load"],label="Load",marker="o",color="lightsalmon") # Plot Load
        axes[j].set_xlabel("Time (hour)")
        axes[j].set_ylabel("Power (kW)")
        axes[j].set_title(day)
        
        axes[j].xaxis.set_major_locator(mdates.HourLocator(interval=2))
        axes[j].xaxis.set_major_formatter(mdates.DateFormatter('%H'))


    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=len(handles))
    fig.tight_layout()
    fig.show()

In [ ]:
def plot_power_curve(df,day,title):
    pos_cols = ["PV",	"Wind", "P_discharge", "P_buy"]
    pos_color = ["gold","lightskyblue","plum","lightgreen"]
    labels = ["PV", "Wind", "Battery", "Grid"]

    neg_cols = ["P_charge","P_sell"]
    neg_color = ["plum","lightgreen"]

    n_cols = 1
    n_rows = 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5, 7))

    axes = axes.flatten()  # Flatten to easily index

    #### PLOT POWER PROFILE ########
    j = 0 # power plot is first in row
    day_plot = df[df["date"] == day]
    SOC = day_plot["SOC"]

    width = 1/len(day_plot) # the width of the bars: can also be len(x) sequence


    top = np.zeros(len(day_plot))
    bottom = np.zeros(len(day_plot))
    for i,col in enumerate(pos_cols): # Plotting all "positive" power
        color = pos_color[i]
        axes[j].bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color = color)

        top += day_plot[col]

    for i,col in enumerate(neg_cols): # Plotting all "negative" power
        color = neg_color[i]
        axes[j].bar(day_plot.index, - day_plot[col], width, bottom=bottom, color=color)
        bottom -= day_plot[col]

    axes[j].step(day_plot.index,day_plot["load"],label="Load",linestyle="--",color="lightsalmon") # Plot Load
    axes[j].set_xlabel("Time (hour)")
    axes[j].set_ylabel("Power (kW)")
    
    axes[j].xaxis.set_major_locator(mdates.HourLocator(interval=2))
    axes[j].xaxis.set_major_formatter(mdates.DateFormatter('%H'))

    ######## PLOT SOC AND PRICE TOGETHER ###############
    j = 1  # Use the same subplot for both

    # Create twin axis for price
    ax_soc = axes[j]
    ax_price = ax_soc.twinx()

    # Plot SOC on left y-axis
    ax_soc.step(day_plot.index, day_plot["SOC"]/E_capacity, color='royalblue', linewidth=2, marker='', label="SOC")
    ax_soc.fill_between(day_plot.index, day_plot["SOC"]/E_capacity, alpha=0.5, color='royalblue',step='pre')
    ax_soc.set_xlabel("Time (hour)")
    ax_soc.set_ylabel("State of Charge", color='royalblue')
    ax_soc.tick_params(axis='y', labelcolor='royalblue')

    # Plot Price on right y-axis
    ax_price.step(day_plot.index, day_plot["price"], color='red', linewidth=2, marker='', label="Day Ahead Price")
    ax_price.set_ylabel("Day Ahead Price (DKK)", color='red')
    ax_price.tick_params(axis='y', labelcolor='red')

    # X-axis formatting
    ax_soc.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax_soc.xaxis.set_major_formatter(mdates.DateFormatter('%H'))

    # Add legends
    ax_soc.legend(loc='upper left')
    ax_price.legend(loc='upper right')


    handles, labels = axes[0].get_legend_handles_labels()    
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=len(handles))
    fig.tight_layout()
    fig.suptitle(title)
    fig.show()

In [ ]:
def compare_power_curve(dfs, day, title=''):
    pos_cols = ["PV", "Wind", "P_discharge", "P_buy"]
    pos_color = ["gold", "lightskyblue", "plum", "lightgreen"]
    labels = ["PV", "Wind", "Battery", "Grid"]

    neg_cols = ["P_charge", "P_sell"]
    neg_color = ["plum", "lightgreen"]

    n_cols = 2
    n_rows = 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 10))
    
    for n in range(2):
        df = dfs[n]
        
        #### PLOT POWER PROFILE (Top row) ########
        ax_power = axes[0, n]  # Row 0, Column n
        day_plot = df[df["date"] == day]
        
        width = 1 / len(day_plot)  # Bar width
        
        top = np.zeros(len(day_plot))
        bottom = np.zeros(len(day_plot))
        
        for i, col in enumerate(pos_cols):
            color = pos_color[i]
            ax_power.bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color=color)
            top += day_plot[col]
        
        for i, col in enumerate(neg_cols):
            color = neg_color[i]
            ax_power.bar(day_plot.index, -day_plot[col], width, bottom=bottom, color=color)
            bottom -= day_plot[col]
        
        ax_power.step(day_plot.index, day_plot["load"], label="Load", linestyle="--", color="lightsalmon", where='pre')
        ax_power.set_xlabel("Time (hour)")
        ax_power.set_ylabel("Power (kW)")
        ax_power.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        ax_power.xaxis.set_major_formatter(mdates.DateFormatter('%H'))
        
        #### PLOT SOC AND PRICE TOGETHER (Bottom row) ########
        ax_soc = axes[1, n]  # Row 1, Column n
        ax_price = ax_soc.twinx()
        
        # Plot SOC on left y-axis
        ax_soc.step(day_plot.index, day_plot["SOC"] / E_capacity, color='royalblue', linewidth=2, 
                    marker='', label="SOC", where='pre')
        ax_soc.fill_between(day_plot.index, day_plot["SOC"] / E_capacity, alpha=0.5, 
                           color='royalblue', step='pre')
        ax_soc.set_xlabel("Time (hour)")
        ax_soc.set_ylabel("State of Charge", color='royalblue')
        ax_soc.tick_params(axis='y', labelcolor='royalblue')
        
        # Plot Price on right y-axis
        ax_price.step(day_plot.index, day_plot["price"], color='red', linewidth=2, 
                      marker='', label="Day Ahead Price", where='pre')
        ax_price.set_ylabel("Day Ahead Price (DKK)", color='red')
        ax_price.tick_params(axis='y', labelcolor='red')
        
        # X-axis formatting
        ax_soc.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        ax_soc.xaxis.set_major_formatter(mdates.DateFormatter('%H'))
        
        # Add legends for this subplot
        ax_soc.legend(loc='upper left')
        ax_price.legend(loc='upper right')
    
    ### Check for largest power and set that as scale 

    all_power = []
    for df in dfs:
        day_plot = df[df["date"] == day]
        pos_cols = ["PV", "Wind", "P_discharge", "P_buy", "load"]
        neg_cols = ["P_charge", "P_sell"]
        all_power.extend([(day_plot[col].min(), day_plot[col].max()) for col in pos_cols if col in day_plot])
        all_power.extend([(-day_plot[col].max(), -day_plot[col].min()) for col in neg_cols if col in day_plot])

    global_min, global_max = np.min(all_power), np.max(all_power)
    padding = (global_max - global_min) * 0.05

    for n in range(2):
        axes[0, n].set_ylim(global_min - padding, global_max + padding)




    # Single legend for power plots (collect from first power subplot)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1), ncol=len(handles))
    
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Optimization for Clustering

In [ ]:
test_days = ['2025-05-17',
 '2025-05-25',
 '2025-05-26',
 '2025-06-12',
 '2025-06-17',
 '2025-06-24',
 '2025-07-03',
 '2025-07-06',
 '2025-07-17',
 '2025-08-09',
 '2025-08-10',
 '2025-08-11',
 '2025-09-08',
 '2025-09-09',
 '2025-09-17',
 '2025-10-11',
 '2025-10-18',
 '2025-10-28',
 '2025-11-04',
 '2025-11-12',
 '2025-11-28',
 '2025-12-09',
 '2025-12-23',
 '2025-12-27',
 '2026-01-22',
 '2026-01-25',
 '2026-01-26',
 '2026-02-19',
 '2026-02-22',
 '2026-02-27',
 '2026-03-04',
 '2026-03-25',
 '2026-03-29',
 '2026-04-06',
 '2026-04-08',
 '2026-04-09']

In [ ]:
# %run daily_profile_clustering_weighted_average.ipynb

In [ ]:
len(test_days)

In [ ]:
pv_cluster_labels = pv_final_result["model_result"]["test_cluster_labels"]
wind_cluster_labels = wind_final_result["model_result"]["test_cluster_labels"]

In [ ]:
wind_final_result["model_result"]["cluster_size_info"]

In [ ]:
df_wind_cluster = wind_final_result["model_result"]["y_test_pred_daily_weighted"]
df_PV_cluster = pv_final_result["model_result"]["y_test_pred_daily_weighted"]

In [ ]:
def pivot_cluster_results(df,value_name,new_col_name):
    # Melt the DataFrame
    df_melted = df.reset_index().melt(
        id_vars='date', 
        var_name='time', 
        value_name=new_col_name
    )

    df_melted['date'] = pd.to_datetime(df_melted['date'])

    # Extract time part from column names
    df_melted['time'] = df_melted['time'].str.replace(value_name, '')

    # Now combine - convert date to string first
    df_melted['ts'] = pd.to_datetime(
        df_melted['date'].dt.strftime('%Y-%m-%d') + ' ' + df_melted['time'])

    # Set datetime as index and drop unnecessary columns
    result_df = df_melted.set_index('ts')
    result_df = result_df.drop(columns="time")
    # Sort by datetime (optional but recommended)
    result_df = result_df.sort_index()
    return result_df

In [ ]:
wind_cluster = pivot_cluster_results(df_wind_cluster,"Wind_total_","Wind_total")
PV_cluster = pivot_cluster_results(df_PV_cluster,"PV_total_","PV_total").drop(columns="date")
df_cluster = PV_cluster.join(wind_cluster,how="inner")
df_cluster = df_cluster.join(df_price,how="inner")
df_cluster = df_cluster.join(total_load_30min.drop(columns="date"),how="inner")
df_cluster_grouped = {str(date)[:-9]: group for date, group in df_cluster.groupby("date")}

In [ ]:
schedule_cluster = operational_schedule(df_cluster_grouped,test_days, E_capacity, P_capacity,charge_eff,discharge_eff)
schedule_oracle = operational_schedule(df_grouped,test_days, E_capacity, P_capacity,charge_eff,discharge_eff)

In [ ]:
plot_like11(schedule_oracle,test_days[:4])

In [ ]:
mean_absolute_error(schedule_oracle["P_charge"],schedule_cluster["P_charge"]) # Barely any error at all

In [ ]:
day = test_days[5]
compare_power_curve([schedule_oracle,schedule_cluster],day,f"{day}")

# Comparison of Oracle vs Prediction

In [ ]:
def real_time_power(oracle,prediction,day,E_capacity, P_capacity, charge_eff, discharge_eff, dT):
    oracle_cols = ["PV", "Wind","P_charge","P_discharge","SOC"]
    cluster_cols = ["P_sell","P_buy","price","load","date"]

    df = prediction[cluster_cols]
    df = df.join(oracle[oracle_cols])
    df = df.rename(columns={"SOC": "SOC_predicted","P_charge":"Pcharge_predicted","P_discharge":"Pdischarge_predicted"})
    df_day = df[df["date"] == day]
    n = len(df_day)

    PV_gen = df_day["PV"].to_numpy()
    Wind_gen = df_day["Wind"].to_numpy()
    load = df_day["load"].to_numpy()
    P_buy = df_day["P_buy"].to_numpy()
    P_sell = df_day["P_sell"].to_numpy()
    price = df_day["price"].to_numpy()

    SOC = np.zeros(n)

    df_day["Imbalances Without Battery"] = PV_gen + Wind_gen + P_buy - load - P_sell
    P_charge = np.where(df_day["Imbalances Without Battery"] > 0, df_day["Imbalances Without Battery"], 0)
    P_discharge = np.where(df_day["Imbalances Without Battery"] < 0, -df_day["Imbalances Without Battery"], 0)

    df_day["Power_imbalance"] = pd.Series(0.0, index=df_day.index)

    for i,date in enumerate(df_day.index):
        # CHECK IF POWER CAPACITY IS MET
        if P_charge[i] > P_capacity:
            df_day.at[date, "Power_imbalance"] = P_charge[i] - P_capacity
            P_charge[i] = P_capacity
        if P_discharge[i] > P_capacity:

            df_day.at[date, "Power_imbalance"] = - P_discharge[i] + P_capacity
            P_discharge[i] = P_capacity

        # Calculate SOC (kWh)
        if i == 0:
            SOC[i] = 0.5 * E_capacity + (P_charge[i] * charge_eff - P_discharge[i] / discharge_eff) * dT
        else:
            SOC[i] = SOC[i-1] + (P_charge[i] * charge_eff - P_discharge[i] / discharge_eff) * dT

        # Check SOC violations, and if needed, Recalculate SOC
        if SOC[i] > E_capacity:
            energy_overshoot = SOC[i] - E_capacity # kWh
            power_overshoot = energy_overshoot /charge_eff / dT #kW
            
            # Add to imbalance
            df_day.at[date, "Power_imbalance"] = df_day.at[date, "Power_imbalance"] + power_overshoot
            
            # Reduce charge power by the overshoot power 
            P_charge[i] = P_charge[i] - power_overshoot
            
            # Reset SOC to max
            SOC[i] = E_capacity

        elif SOC[i] < 0.1 * E_capacity:
            # Energy undershoot (kWh)
            energy_undershoot = 0.1 * E_capacity - SOC[i]
            power_undershoot = energy_undershoot*discharge_eff / dT
            # Add to imbalance
            df_day.at[date, "Power_imbalance"] = df_day.at[date, "Power_imbalance"] - power_undershoot
            
            # Adjust P_discharge 
            P_discharge[i] = P_discharge[i] - power_undershoot
            
            # Reset SOC to min
            SOC[i] = 0.1 * E_capacity
            
    df_day["SOC"] = SOC
    df_day["Pcharge"] = P_charge
    df_day["Pdischarge"] = P_discharge

    too_much_energy = sum(df_day.loc[df_day["Power_imbalance"]>0,"Power_imbalance"])*dT
    too_little_energy = sum(df_day.loc[df_day["Power_imbalance"]<0,"Power_imbalance"])*dT
    
    df_day["Total_E_deficit"] = too_little_energy
    df_day["Total_E_overspil"] = too_much_energy
    return df_day

In [ ]:
schedule_realtime = real_time_power(schedule_oracle,schedule_cluster,"2025-06-24",E_capacity, P_capacity, charge_eff, discharge_eff, dT)

In [ ]:
schedule_realtime[["SOC","Pcharge","Pdischarge","Imbalances Without Battery","Power_imbalance"]]

In [ ]:
def real_time_power_WRONG(oracle,prediction,day,E_capacity, P_capacity, charge_eff, discharge_eff, dT, C_rate,solver=cp.OSQP):
    #################### Pre Processing ########################
    oracle_cols = ["PV", "Wind","P_charge","P_discharge","SOC"]
    cluster_cols = ["P_sell","P_buy","price","load","date"]

    df = prediction[cluster_cols]
    df = df.join(oracle[oracle_cols])
    df = df.rename(columns={"SOC": "SOC_predicted","P_charge":"Pcharge_predicted","P_discharge":"Pdischarge_predicted"})
    df_day = df[df["date"] == day]
    n = len(df_day)

    PV_gen = df_day["PV"].to_numpy()
    Wind_gen = df_day["Wind"].to_numpy()
    load = df_day["load"].to_numpy()
    P_buy = df_day["P_buy"].to_numpy()
    P_sell = df_day["P_sell"].to_numpy()
    price = df_day["price"].to_numpy()

    # creating an extra column for the new SOC, P_charge and P_discharge
    df_day["SOC"] = df_day["SOC_predicted"] 
    df_day["P_charge"] = df_day["Pcharge_predicted"]
    df_day["P_discharge"] = df_day["Pdischarge_predicted"]
    df_day["accumulated_power without Battery"] = df_day["PV"] + df_day["Wind"] + df_day["P_buy"] - df_day["load"] - df_day["P_sell"] 
    ############################################################

    #################### Defining Problem ####################
    Pload = cp.Constant(load)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)
    P_capacity = cp.Constant(P_capacity)
    E_capacity = cp.Constant(E_capacity)
    Pgrid_buy = cp.Constant(P_buy) # if positive, energy is being bought from the grid. if negative we are selling energy to the grid. 
    Pgrid_sell = cp.Constant(P_sell)


    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)

    P_inbalance = cp.Variable(n) # If positive we have too much energy, if negative we have too little
    SOC = cp.Variable(n)

    constraints = [
        Pcharge <= P_capacity,
        Pdischarge <= P_capacity,

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy - Pgrid_sell == Pload + Pcharge + P_inbalance,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        #SOC[n - 1] >= 0.5 * E_capacity # Do we want this?  
        ]
    
    
    regularization = 1e-5 * (cp.sum_squares(Pcharge) + cp.sum_squares(Pdischarge)) ## For reference, has the value: 0.3683 for test_days[3]
    problem = cp.Problem(cp.Minimize(cp.sum(cp.abs(P_inbalance))+regularization), constraints)
    

    try:
        problem.solve(solver=solver, verbose=False,max_iter=10000000)

        P_battery = Pcharge-Pdischarge
        Pcharge_total = cp.pos(P_battery)
        Pdischarge_total = cp.neg(P_battery)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"Problem arrose: {problem.status}")
            return df_day,{
            "status": problem.status,
            "Power_imbalance": np.nan,
            "Total_E_deficit": np.nan,
            "Total_E_overspil": np.nan,

            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
            }
        
        P_deficit = cp.neg(P_inbalance)
        P_overspil = cp.pos(P_inbalance)

        E_deficit = cp.sum(P_deficit*dT)
        E_overspil = cp.sum(P_overspil*dT)

        return df_day,{
        
            "status": problem.status,

            "Power_imbalance": P_inbalance.value,
            "Total_E_deficit": E_deficit.value,
            "Total_E_overspil": E_overspil.value,
            "regularization": regularization.value,

            "Pcharge": Pcharge_total.value,
            "Pdischarge": Pdischarge_total.value,
            "SOC": SOC.value
        }

    except Exception as e:
        print(f"Error arose {type(e).__name__}")
        return df_day,{
            "status": type(e).__name__,
            "Power_imbalance": np.nan,
            "Total_E_deficit": np.nan,
            "Total_E_overspil": np.nan,

            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
        }
    

In [ ]:
def run_realtime_power_for_all_days(oracle,prediction,days:list,E_capacity, P_capacity, charge_eff, discharge_eff, dT):
    profiles = []
    for day in days:
        print("---------------------------------------------------------")
        print("Realtime Schedule starting for ", day)
        print("---------------------------------------------------------")

        day_df = real_time_power(oracle,
                              prediction,
                              day,
                              E_capacity,
                              P_capacity,
                              charge_eff,
                              discharge_eff,
                              dT
                            
                              
                            )

        # Build time-series dataframe
        profile_df = pd.DataFrame({
            "ts": day_df.index,
            "Power_imbalance": day_df["Power_imbalance"],
            "Total_Daily_E_deficit": day_df["Total_E_deficit"],
            "Total_Daily_E_overspil": day_df["Total_E_overspil"],
            "P_charge": day_df["Pcharge"],
            "P_discharge": day_df["Pdischarge"],
            "Pcharge_predicted": day_df["Pcharge_predicted"].values,
            "Pdischarge_predicted": day_df["Pdischarge_predicted"].values,
            "Imbalances Without Battery": day_df["Imbalances Without Battery"].values,
            "SOC": day_df["SOC"],
            "SOC_predicted": day_df["SOC_predicted"].values,
            "price": day_df["price"].values,
            "load": day_df["load"].values,
            "PV": day_df["PV"].values,
            "Wind": day_df["Wind"].values,
            "P_buy": day_df["P_buy"].values,
            "P_sell": day_df["P_sell"].values,
        
        })

        profile_df["date"] = day
        profiles.append(profile_df)


    profiles_df = pd.concat(profiles, ignore_index=True)
    profiles_df = profiles_df.set_index("ts")
    return profiles_df

In [ ]:
schedule_realtime

In [ ]:
# Run the Optimization for the given days
days = test_days
schedule_realtime = run_realtime_power_for_all_days(schedule_oracle,schedule_cluster,days,E_capacity, P_capacity, charge_eff, discharge_eff, dT)

In [ ]:
def real_time_power_curve(df,day,title=" "):
    pos_cols = ["PV",	"Wind", "Pdischarge", "P_buy"]
    pos_color = ["gold","lightskyblue","plum","lightgreen"]
    labels = ["PV", "Wind", "Battery", "Grid"]

    neg_cols = ["Pcharge","P_sell"]
    neg_color = ["plum","lightgreen"]

    n_cols = 1
    n_rows = 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5, 7))

    axes = axes.flatten()  # Flatten to easily index

    #### PLOT POWER PROFILE ########
    j = 0 # power plot is first in row
    day_plot = df[df["date"] == day]
    SOC = day_plot["SOC"]

    width = 1/len(day_plot) # the width of the bars: can also be len(x) sequence


    top = np.zeros(len(day_plot))
    bottom = np.zeros(len(day_plot))
    for i,col in enumerate(pos_cols): # Plotting all "positive" power
        color = pos_color[i]
        axes[j].bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color = color)

        top += day_plot[col]

    for i,col in enumerate(neg_cols): # Plotting all "negative" power
        color = neg_color[i]
        axes[j].bar(day_plot.index, - day_plot[col], width, bottom=bottom, color=color)
        bottom -= day_plot[col]

    axes[j].step(day_plot.index,day_plot["load"],label="Load",linestyle="--",color="lightsalmon") # Plot Load
    axes[j].step(day_plot.index,day_plot["Power_imbalance"],label="Power imbalance",linestyle="-",color="red") # Plot Load
    axes[j].set_xlabel("Time (hour)")
    axes[j].set_ylabel("Power (kW)")
    
    axes[j].xaxis.set_major_locator(mdates.HourLocator(interval=2))
    axes[j].xaxis.set_major_formatter(mdates.DateFormatter('%H'))

    ######## PLOT SOC AND PRICE TOGETHER ###############
    j = 1  # Use the same subplot for both

    # Create twin axis for price
    ax_soc = axes[j]
    ax_price = ax_soc.twinx()

    # Plot SOC on left y-axis
    ax_soc.step(day_plot.index, day_plot["SOC_predicted"]/E_capacity, color='royalblue', linewidth=2, marker='', label="Planned SOC")
    ax_soc.fill_between(day_plot.index, day_plot["SOC_predicted"]/E_capacity, alpha=0.5, color='royalblue',step='pre')

    ax_soc.step(day_plot.index, day_plot["SOC"]/E_capacity, color='salmon', linewidth=2, marker='', label="SOC")
    ax_soc.fill_between(day_plot.index, day_plot["SOC"]/E_capacity, alpha=0.5, color='salmon',step='pre')

    ax_soc.set_xlabel("Time (hour)")
    ax_soc.set_ylabel("State of Charge", color='royalblue')
    ax_soc.tick_params(axis='y', labelcolor='royalblue')

    # Plot Price on right y-axis
    ax_price.step(day_plot.index, day_plot["price"], color='red', linewidth=2, marker='', label="Day Ahead Price")
    ax_price.set_ylabel("Day Ahead Price (DKK)", color='red')
    ax_price.tick_params(axis='y', labelcolor='red')

    # X-axis formatting
    ax_soc.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax_soc.xaxis.set_major_formatter(mdates.DateFormatter('%H'))

    # Add legends
    ax_soc.legend(loc='upper left')
    ax_price.legend(loc='upper right')


    handles, labels = axes[0].get_legend_handles_labels()    
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=len(handles))
    fig.tight_layout()
    fig.suptitle(title)
    fig.show()

In [ ]:
def compare_all_power_curves(df_oracle, df_prediction, df_real_time, day, title='',fontsize=14):
    pos_cols = ["PV", "Wind", "P_discharge", "P_buy"]
    pos_color = ["gold", "lightskyblue", "plum", "lightgreen"]
    labels = ["PV", "Wind", "Battery", "Grid"]

    neg_cols = ["P_charge", "P_sell"]
    neg_color = ["plum", "lightgreen"]

    n_cols = 3
    n_rows = 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 10))
    
    # List of (df, name) for each column
    scenarios = [
        (df_prediction, "Prediction from Clustering"),
        (df_oracle, "Oracle"),
        (df_real_time, "Real Time")
    ]
    
    for n, (df, name) in enumerate(scenarios):
        day_plot = df[df["date"] == day] if "date" in df.columns else df
        
        #### PLOT POWER PROFILE (Top row) ########
        ax_power = axes[0, n]
        
        width = 0.85 / len(day_plot)
        
        top = np.zeros(len(day_plot))
        bottom = np.zeros(len(day_plot))
        
        for i, col in enumerate(pos_cols):
            if col in day_plot.columns:
                color = pos_color[i]
                ax_power.bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color=color)
                top += day_plot[col].values
        
        for i, col in enumerate(neg_cols):
            if col in day_plot.columns:
                color = neg_color[i]
                ax_power.bar(day_plot.index, -day_plot[col], width, bottom=bottom, color=color)
                bottom -= day_plot[col].values
        
        # Plot Load
        if "load" in day_plot.columns:
            ax_power.step(day_plot.index, day_plot["load"], label="Load", linestyle="--", 
                         color="orange", where='pre',linewidth=3)
        
        # Plot Power Imbalance (only for real_time)
        if name == "Real Time" and "Power_imbalance" in day_plot.columns:
            ax_power.step(day_plot.index, day_plot["Power_imbalance"], label="Power Imbalance", 
                         linestyle="-", color="red", where='pre')
        
        ax_power.set_xlabel("Time (hour)",fontsize=fontsize)
        ax_power.set_ylabel("Power (kW)",fontsize=fontsize)
        ax_power.set_title(f"{name}",fontsize=fontsize)
        ax_power.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        ax_power.xaxis.set_major_formatter(mdates.DateFormatter('%H'))
        
        #### PLOT SOC AND PRICE TOGETHER (Bottom row) ########
        ax_soc = axes[1, n]
        ax_price = ax_soc.twinx()
        
        # Plot SOC on left y-axis

        if "SOC" in day_plot.columns:
            ax_soc.step(day_plot.index, day_plot["SOC"] / E_capacity, 
                        color='royalblue', linewidth=2, marker='', label="SOC", where='pre')
            ax_soc.fill_between(day_plot.index, day_plot["SOC"] / E_capacity, 
                                alpha=0.5, color='royalblue', step='pre')
        
        ax_soc.set_xlabel("Time (hour)",fontsize=fontsize)
        ax_soc.set_ylabel("State of Charge", color='royalblue',fontsize=fontsize)
        ax_soc.tick_params(axis='y', labelcolor='royalblue')
        
        # Plot Price on right y-axis
        if "price" in day_plot.columns:
            ax_price.step(day_plot.index, day_plot["price"], color='red', linewidth=2, 
                         marker='', label="Day Ahead Price", where='pre')
            ax_price.set_ylabel("Day Ahead Price (DKK)", color='red',fontsize=fontsize)
            ax_price.tick_params(axis='y', labelcolor='red')
        
        # X-axis formatting
        ax_soc.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        ax_soc.xaxis.set_major_formatter(mdates.DateFormatter('%H'))
        
        # Add legends for this subplot
        ax_soc.legend(loc='upper left')
        ax_price.legend(loc='upper right')
    
    # Set same y-scale for all power plots
    all_power = []
    for n, (df, name) in enumerate(scenarios):
        day_plot = df[df["date"] == day] if "date" in df.columns else df
        pos_cols_list = ["PV", "Wind", "P_discharge", "P_buy", "load"]
        neg_cols_list = ["P_charge", "P_sell", "Power_imbalance"]
        
        for col in pos_cols_list:
            if col in day_plot.columns:
                all_power.extend([day_plot[col].min(), day_plot[col].max()])
        for col in neg_cols_list:
            if col in day_plot.columns:
                all_power.extend([-day_plot[col].max(), -day_plot[col].min()])
    
    if all_power:
        global_min, global_max = np.min(all_power), np.max(all_power)
        padding = abs(global_max - global_min) * 0.2
        for n in range(3):
            axes[0, n].set_ylim(global_min - padding, global_max + padding)
    
    # Single legend for power plots
    handles, labels = ax_power.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=len(handles),fontsize=14)
    
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
test_days = ['2025-05-17',
 '2025-05-25',
 '2025-05-26',
 '2025-06-12',
 '2025-06-17',
 '2025-06-24',
 '2025-07-03',
 '2025-07-06',
 '2025-07-17',
 '2025-08-09',
 '2025-08-10',
 '2025-08-11',
 '2025-09-08',
 '2025-09-09',
 '2025-09-17',
 '2025-10-11',
 '2025-10-18',
 '2025-10-28',
 '2025-11-04',
 '2025-11-12',
 '2025-11-28',
 '2025-12-09',
 '2025-12-23',
 '2025-12-27',
 '2026-01-22',
 '2026-01-25',
 '2026-01-26',
 '2026-02-19',
 '2026-02-22',
 '2026-02-27',
 '2026-03-25', #### REMOVED 03-04-2026
 '2026-03-29',
 '2026-04-06',
 '2026-04-08',
 '2026-04-09']

In [ ]:
# Plot Power Curves for a few days
days = ["2025-10-11","2026-02-22","2025-06-24",'2025-05-17',"2026-04-09"]
#days = [test_days[0]]
for test_day in days:
    compare_all_power_curves(schedule_oracle,schedule_cluster,schedule_realtime,test_day,f"{test_day}",fontsize=20)

In [ ]:
print(schedule_realtime[["Total_E_overspil","Total_E_deficit"]])

In [ ]:
av_daily_negative_imbalance = sum(schedule_realtime.loc[schedule_realtime["Power_imbalance"]<0,"Power_imbalance"])*0.5/(len(schedule_realtime)/48) # kWh 
av_daily_positive_imbalance = sum(schedule_realtime.loc[schedule_realtime["Power_imbalance"]>0,"Power_imbalance"])*0.5/(len(schedule_realtime)/48) # kWh
av_daily_total_imbalance = sum(schedule_realtime["Power_imbalance"])*0.5/(len(schedule_realtime)/48) # kWh
av_daily_negative_imbalance,av_daily_positive_imbalance, av_daily_total_imbalance

In [ ]:
daily_negative_imbalance = []
daily_positive_imbalance = []
pv_results = {0:0,1:0,2:0,3:0,4:0}
wind_results = {0:0,1:0,2:0,3:0,4:0}
imbalance_results = pd.DataFrame()
imbalance_results = imbalance_results.assign(date=test_days).set_index("date")
imbalance_results["Wind Cluster"] = wind_cluster_labels
imbalance_results["PV Cluster"] = pv_cluster_labels


for i,day in enumerate(test_days): 
    neg_imbalance = sum(schedule_realtime.loc[(schedule_realtime["Power_imbalance"]<0)&(schedule_realtime["date"]==day),"Power_imbalance"])*0.5
    pos_imbalance = sum(schedule_realtime.loc[(schedule_realtime["Power_imbalance"]>0)&(schedule_realtime["date"]==day),"Power_imbalance"])*0.5
    imbalance_results.at[day,"E Deficit"] = neg_imbalance
    imbalance_results.at[day,"E Surplus"] = pos_imbalance
    imbalance_results.at[day,"E Imbalance Total"] = pos_imbalance - neg_imbalance
W_res = pd.DataFrame()
PV_res = pd.DataFrame()
both_res = pd.DataFrame(dtype=None)
cols = ["E Deficit","E Surplus","E Imbalance Total"]
for i in range(5):
    for col in cols: 
        W_res.at[i,col] = imbalance_results.loc[imbalance_results["Wind Cluster"]==i,col].mean()
        PV_res.at[i,col] = imbalance_results.loc[imbalance_results["PV Cluster"]==i,col].mean()
    for j in range(5):
        for col in cols: 
            both_res.at[f"({i,j})",col] = imbalance_results.loc[(imbalance_results["PV Cluster"]==i) & (imbalance_results["Wind Cluster"]==j),col].mean()
# ORDER IS (PV cluster, Wind cluster)

In [ ]:
both_res

In [ ]:
cluster_

In [ ]:
W_res

In [ ]:
PV_res

In [ ]:
imbalance_results

In [ ]:
def size_battery(df_grouped,df_cluster_grouped,energy_range:list,power_range:list,test_days):
    out = {}
    for E_capacity,P_capacity in zip(energy_range,power_range):
        schedule_cluster = operational_schedule(df_cluster_grouped,test_days, E_capacity, P_capacity,charge_eff,discharge_eff)
        schedule_oracle = operational_schedule(df_grouped,test_days, E_capacity, P_capacity,charge_eff,discharge_eff)
        schedule_realtime = run_realtime_power_for_all_days(schedule_oracle,schedule_cluster,test_days,E_capacity,P_capacity,charge_eff,discharge_eff,dT)
        av_daily_negative_imbalance = sum(schedule_realtime.loc[schedule_realtime["Power_imbalance"]<0,"Power_imbalance"])*0.5/(len(schedule_realtime)/48) # kWh 
        av_daily_positive_imbalance = sum(schedule_realtime.loc[schedule_realtime["Power_imbalance"]>0,"Power_imbalance"])*0.5/(len(schedule_realtime)/48) # kWh
        out[(E_capacity,P_capacity)] = (av_daily_negative_imbalance,av_daily_positive_imbalance)
    return out

In [ ]:
power_range = [1,30,60,90]
energy_range = [6,120,160,300]
energy_power = size_battery(df_grouped,df_cluster_grouped,energy_range,power_range,test_days)